# TrustOCT: A Trustworthy-AI Evaluation Framework for OCT Retinal Disease Classification
### Reference model: ResNet50 + Multi-Scale Feature Fusion (MSF) + CBAM, evaluated on Kermany/Mendeley OCT2017

**M.Tech thesis / journal-paper project notebook — designed to run end-to-end on Google Colab (free or Pro GPU runtime).**

**TrustOCT is the evaluation framework, not the CNN.** `ResNetMSFCBAM` is one
reference model used to demonstrate the framework; MSF/CBAM are a standard
fusion + attention combination chosen to demonstrate the framework end to end,
not claimed as a novel architecture in their own right. The paper's
contribution claim should rest on the evaluation methodology (Phases 3–5
below), not on the backbone.

**What this notebook does, in order:**
1. Environment setup + Kaggle dataset download (Kermany OCT2017: CNV / DME / DRUSEN / NORMAL)
2. Writes the `trustoct` Python package to disk (same code you'd put in your thesis appendix / GitHub repo)
3. **Phase 1** — Baseline: `EXP001` plain ResNet50
4. **Phase 2** — Ablation: `EXP002` (+MSF), `EXP003` (+MSF+CBAM) → one metrics table (accuracy, precision, recall, specificity, macro-F1, balanced accuracy, MCC, Cohen's Kappa, ROC-AUC)
5. **Phase 3 (core contribution)** — Calibration (ECE, Brier, avg confidence vs. avg accuracy, reliability diagrams) + Explainability faithfulness (LayerCAM + Deletion/Insertion AOPC)
6. **Phase 4 (supporting)** — Robustness to noise/blur/brightness/contrast, compute cost analysis, failure-case gallery
7. **Phase 5 (credibility additions)** — Multi-seed statistical significance testing, and optional external-dataset validation for generalization evidence

> **Before you start:** Runtime → Change runtime type → **GPU** (T4 is fine). Upload your `kaggle.json` when prompted in Section 1, or set `KAGGLE_USERNAME`/`KAGGLE_KEY` as Colab secrets.

> **Data integrity check:** if any validation accuracy looks suspiciously
> perfect (e.g. 100% in epoch 1), stop and re-run `assert_no_patient_leakage`
> on the exact split in use before trusting the number — see the README's
> "Notes on reproducibility" section.


## 1. Environment setup

In [ ]:
!pip install -q kagglehub opencv-python-headless
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected — go to Runtime > Change runtime type > GPU before training.")


### 1.1 Kaggle authentication
Two options:
- **Upload `kaggle.json`** (from kaggle.com → Account → Create New API Token) using the file picker below, OR
- Skip the upload cell and instead set Colab secrets `KAGGLE_USERNAME` / `KAGGLE_KEY` (key icon in the left sidebar) — kagglehub will pick them up automatically.


In [ ]:
import os
# Option A: upload kaggle.json interactively (skip this cell if using Colab secrets instead)
try:
    from google.colab import files
    if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
        print("Upload your kaggle.json (Kaggle account -> Create New API Token):")
        uploaded = files.upload()
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        for fname in uploaded:
            os.rename(fname, os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        print("Saved kaggle.json")
    else:
        print("kaggle.json already present.")
except ImportError:
    print("Not running on Colab — set KAGGLE_USERNAME / KAGGLE_KEY env vars manually.")


## 2. Write the `trustoct` package to disk
This mirrors the exact package structure you should also keep in your GitHub repo / thesis code appendix:
```
trustoct/
  __init__.py
  utils.py            # seeding, checkpointing
  data.py             # Kermany loading, CLAHE, stratified split, Dataset
  modules.py          # CBAM, MSF building blocks
  model.py            # TrustOCTNet (ResNet50 + optional MSF + optional CBAM)
  train.py            # shared training loop for all 3 experiments
  metrics.py          # ablation metrics table
  calibration.py      # ECE, Brier, reliability diagrams
  explainability.py   # LayerCAM + Deletion/Insertion AOPC
  robustness.py       # noise/blur/brightness/contrast perturbation eval
```


In [ ]:
import os
os.makedirs('trustoct', exist_ok=True)

In [ ]:
%%writefile trustoct/__init__.py
"""
TrustOCT — a trustworthy-AI evaluation framework for OCT retinal disease
classification (CNV / DME / DRUSEN / NORMAL), built around ResNet50 + MSF + CBAM.

Not just "another model" — the contribution is the evaluation methodology:
accuracy is necessary but not sufficient, so this package also measures
calibration (ECE, Brier), explanation faithfulness (LayerCAM + Deletion/
Insertion AOPC), and robustness to acquisition-noise perturbations.
"""

from trustoct.model import (
    build_model, ResNetMSFCBAM, TrustOCTNet, EXPERIMENTS,
    build_resnet50, build_resnet50_msf, build_resnet50_msf_cbam,
)
from trustoct.data import (
    OCTDataset, build_transforms, index_kermany_folder, stratified_split,
    patient_grouped_stratified_split, assert_no_patient_leakage, extract_patient_id,
    download_kermany_dataset, CLASSES,
)
from trustoct.train import fit, compute_class_weights
from trustoct.metrics import get_predictions, compute_metrics, build_ablation_table
from trustoct.calibration import calibration_report, plot_reliability_diagram
from trustoct.explainability import LayerCAM, faithfulness_report, plot_cam_grid
from trustoct.robustness import evaluate_under_perturbation
from trustoct.utils import set_seed, get_device, count_parameters
from trustoct.multiseed import run_multiseed_ablation, run_single_seed, format_mean_std_table
from trustoct.external_validation import run_external_validation, index_external_folder

__all__ = [
    "build_model", "ResNetMSFCBAM", "TrustOCTNet", "EXPERIMENTS",
    "build_resnet50", "build_resnet50_msf", "build_resnet50_msf_cbam",
    "OCTDataset", "build_transforms", "index_kermany_folder", "stratified_split",
    "patient_grouped_stratified_split", "assert_no_patient_leakage", "extract_patient_id",
    "download_kermany_dataset", "CLASSES",
    "fit", "compute_class_weights",
    "get_predictions", "compute_metrics", "build_ablation_table",
    "calibration_report", "plot_reliability_diagram",
    "LayerCAM", "faithfulness_report", "plot_cam_grid",
    "evaluate_under_perturbation",
    "set_seed", "get_device", "count_parameters",
    "run_multiseed_ablation", "run_single_seed", "format_mean_std_table",
    "run_external_validation", "index_external_folder",
]


In [ ]:
%%writefile trustoct/utils.py
"""
trustoct.utils
Reproducibility and misc helpers.
"""
import os
import random
import numpy as np
import torch


def set_seed(seed: int = 42):
    """Fix all relevant seeds. Call this ONCE at the top of every script/notebook cell
    that trains or evaluates a model, so runs are reproducible and defensible in a viva."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_params": total, "trainable_params": trainable}


def save_checkpoint(model, optimizer, epoch, best_metric, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_metric": best_metric,
    }, path)


def load_checkpoint(model, path, optimizer=None, map_location=None):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model_state"])
    if optimizer is not None and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    return ckpt.get("epoch", 0), ckpt.get("best_metric", None)


class AverageMeter:
    """Tracks running average of a metric (loss, acc, etc.) within an epoch."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0.0
        self.count = 0

    def update(self, val, n=1):
        self.sum += val * n
        self.count += n

    @property
    def avg(self):
        return self.sum / max(self.count, 1)


In [ ]:
%%writefile trustoct/modules.py
"""
trustoct.modules
Architectural building blocks: CBAM (Convolutional Block Attention Module,
Woo et al. 2018) and MSF (Multi-Scale Feature fusion module).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# CBAM
# ---------------------------------------------------------------------------
class ChannelAttention(nn.Module):
    """Squeezes spatial dims via avg+max pool, learns a shared MLP over both,
    sums, sigmoids -> per-channel weight. Tells the network 'which feature maps
    matter', e.g. layer-thickness channels vs texture channels."""

    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        hidden = max(in_channels // reduction_ratio, 8)
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, in_channels, bias=False),
        )
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):
        b, c, _, _ = x.shape
        avg_out = self.mlp(self.avg_pool(x).view(b, c))
        max_out = self.mlp(self.max_pool(x).view(b, c))
        attn = torch.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * attn


class SpatialAttention(nn.Module):
    """Squeezes channel dim via avg+max pool, convolves the 2-channel map with a
    7x7 kernel, sigmoids -> per-pixel weight. Tells the network 'where to look',
    e.g. the retinal layer boundary region rather than background."""

    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        attn = torch.sigmoid(self.conv(concat))
        return x * attn


class CBAM(nn.Module):
    """Sequential channel-then-spatial attention, as in the original paper.
    Drop-in module: output has identical shape to input."""

    def __init__(self, in_channels, reduction_ratio=16, spatial_kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(in_channels, reduction_ratio)
        self.spatial_attn = SpatialAttention(spatial_kernel_size)

    def forward(self, x):
        x = self.channel_attn(x)
        x = self.spatial_attn(x)
        return x


# ---------------------------------------------------------------------------
# MSF - Multi-Scale Feature fusion
# ---------------------------------------------------------------------------
class MSFModule(nn.Module):
    """Fuses feature maps from multiple ResNet stages (e.g. layer2, layer3, layer4)
    at a common spatial resolution and channel width, via 1x1 projection + upsample
    + concatenation + 3x3 fusion conv. Motivation for OCT specifically: pathology
    (fluid pockets in DME, neovascular membranes in CNV, drusen deposits) appears at
    very different physical scales in a B-scan, so a single-resolution feature map
    from only the last ResNet stage can miss small/early-stage lesions.
    """

    def __init__(self, in_channels_list, out_channels=256):
        """in_channels_list: channel counts of the feature maps to fuse, ordered from
        shallow -> deep, e.g. [512, 1024, 2048] for ResNet50 layer2/3/4."""
        super().__init__()
        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            ) for c in in_channels_list
        ])
        self.fuse_conv = nn.Sequential(
            nn.Conv2d(out_channels * len(in_channels_list), out_channels,
                      kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, feature_maps):
        """feature_maps: list of tensors [B, C_i, H_i, W_i], shallow->deep.
        All are projected to out_channels and upsampled to the shallowest map's
        spatial size before fusion."""
        target_size = feature_maps[0].shape[-2:]
        projected = []
        for feat, proj in zip(feature_maps, self.projections):
            p = proj(feat)
            if p.shape[-2:] != target_size:
                p = F.interpolate(p, size=target_size, mode="bilinear", align_corners=False)
            projected.append(p)
        fused = torch.cat(projected, dim=1)
        return self.fuse_conv(fused)


In [ ]:
%%writefile trustoct/data.py
"""
trustoct.data
Kermany/Mendeley OCT2017 dataset loading for Colab.

Dataset: "Labeled Optical Coherence Tomography (OCT) and Chest X-Ray Images for
Classification" (Kermany et al., 2018) — CNV / DME / DRUSEN / NORMAL, ~84,000 train
images + the original test/val split.

On Colab, the easiest reliable path is Kaggle via kagglehub:
    kaggle dataset: paultimothymooney/kermany2018
"""
import os
import re
import glob
import random
import cv2
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms

CLASSES = ["NORMAL", "CNV", "DME", "DRUSEN"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def download_kermany_dataset(dest_dir="/content/data"):
    """Downloads OCT2017 dataset on Colab using kagglehub.
    Requires kaggle.json uploaded / KAGGLE credentials set, OR kagglehub's
    interactive auth. Run this once per Colab session.
    """
    import kagglehub
    path = kagglehub.dataset_download("paultimothymooney/kermany2018")
    print(f"Dataset downloaded to: {path}")
    return path


def apply_clahe(img_np: np.ndarray, clip_limit=2.0, tile_grid_size=(8, 8)) -> np.ndarray:
    """Contrast-Limited Adaptive Histogram Equalization — standard OCT preprocessing
    step to boost layer contrast before feeding into a network pretrained on natural
    images. Operates on single-channel (grayscale) OCT B-scans."""
    if img_np.ndim == 3:
        img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    out = clahe.apply(img_np.astype(np.uint8))
    return cv2.cvtColor(out, cv2.COLOR_GRAY2RGB)


class CLAHETransform:
    """torchvision-compatible transform wrapping apply_clahe, operates on PIL Image."""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, pil_img: Image.Image) -> Image.Image:
        arr = np.array(pil_img.convert("L"))
        out = apply_clahe(arr, self.clip_limit, self.tile_grid_size)
        return Image.fromarray(out)


def build_transforms(image_size=224, train=True, use_clahe=True):
    ops = []
    if use_clahe:
        ops.append(CLAHETransform())
    if train:
        ops += [
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
        ]
    else:
        ops += [transforms.Resize((image_size, image_size))]
    ops += [
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    return transforms.Compose(ops)


class OCTDataset(Dataset):
    """Generic ImageFolder-style dataset for CNV/DME/DRUSEN/NORMAL, built from an
    explicit list of file paths so we control the exact train/val/test split
    (important: Kermany's own 'test' folder has only 8 images/class — too small for
    a stable test metric, so we re-split the ~84k 'train' folder ourselves)."""

    def __init__(self, filepaths, labels, transform=None):
        assert len(filepaths) == len(labels)
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        label = self.labels[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label, path


def index_kermany_folder(root_train_dir):
    """Scans OCT2017/train/{CLASS}/*.jpeg and returns (filepaths, labels)."""
    filepaths, labels = [], []
    for cls in CLASSES:
        cls_dir = os.path.join(root_train_dir, cls)
        files = sorted(glob.glob(os.path.join(cls_dir, "*.jpeg")) +
                        glob.glob(os.path.join(cls_dir, "*.jpg")) +
                        glob.glob(os.path.join(cls_dir, "*.png")))
        filepaths += files
        labels += [CLASS_TO_IDX[cls]] * len(files)
    return filepaths, labels


_PATIENT_ID_PATTERN = re.compile(r"^([A-Za-z]+)-(\d+)-\d+\.\w+$")


def extract_patient_id(filepath):
    """Kermany filenames encode a patient ID: e.g. 'CNV-1016042-1.jpeg' ->
    class=CNV, patient_id=1016042, image_index=1. A single patient contributes
    MULTIPLE B-scans, so splitting by image (not patient) lets the same
    patient's scans leak across train/val/test — inflating reported accuracy,
    since adjacent B-scans from one eye are highly correlated. This extracts
    the patient ID so splitting can be done at the patient level instead.

    Falls back to 'parent_dir/basename' (i.e. treats every image as its own
    'patient', namespaced by its containing folder so fallback IDs can't
    collide across classes) if the filename doesn't match the expected
    pattern — this keeps the pipeline from crashing on an unexpected naming
    scheme, but prints a one-time warning since it means leakage protection
    is NOT actually active for those files.
    """
    basename = os.path.basename(filepath)
    m = _PATIENT_ID_PATTERN.match(basename)
    if m:
        return m.group(2)  # numeric patient ID
    parent_dir = os.path.basename(os.path.dirname(filepath))
    return f"{parent_dir}/{basename}"  # unrecognized pattern -> no real grouping, but namespaced


_warned_ungrouped = False


def patient_grouped_stratified_split(filepaths, labels, val_frac=0.10, test_frac=0.10,
                                      seed=42, max_per_class=None):
    """Stratified split by class AND grouped by patient, so no patient's images
    appear in more than one of train/val/test. This is the split you should
    use for any number you intend to report or publish — a plain per-image
    split (see `stratified_split` below, kept only for quick experimentation)
    silently inflates test accuracy via patient leakage.

    Algorithm per class: group filepaths by patient ID, shuffle the patient
    groups (not the individual images), then greedily assign whole patient
    groups to test -> val -> train until each split's image-count target is
    reached. Patients (not images) are the unit being split, so the final
    image counts will approximate but not exactly hit val_frac/test_frac
    (patients have different numbers of scans).

    max_per_class: caps images per class by keeping whole patient groups
    (never splits a patient's images across the cap boundary) up to
    approximately max_per_class images.
    """
    global _warned_ungrouped
    rng = random.Random(seed)

    # class -> patient_id -> [filepaths]
    by_class_patient = {i: {} for i in range(len(CLASSES))}
    ungrouped_count = 0
    for fp, lb in zip(filepaths, labels):
        pid = extract_patient_id(fp)
        basename = os.path.basename(fp)
        if not _PATIENT_ID_PATTERN.match(basename):
            ungrouped_count += 1
        by_class_patient[lb].setdefault(pid, []).append(fp)

    if ungrouped_count > 0 and not _warned_ungrouped:
        print(f"WARNING: {ungrouped_count} filenames didn't match the expected "
              f"Kermany 'CLASS-patientID-index.ext' pattern and could not be "
              f"grouped by patient — leakage protection is NOT active for them. "
              f"Inspect a few filenames if this number is large.")
        _warned_ungrouped = True

    train_fp, train_lb = [], []
    val_fp, val_lb = [], []
    test_fp, test_lb = [], []
    patient_counts = {"train": 0, "val": 0, "test": 0}

    for cls_idx, patient_dict in by_class_patient.items():
        patient_ids = list(patient_dict.keys())
        rng.shuffle(patient_ids)

        if max_per_class is not None:
            capped_ids, running_total = [], 0
            for pid in patient_ids:
                if running_total >= max_per_class:
                    break
                capped_ids.append(pid)
                running_total += len(patient_dict[pid])
            patient_ids = capped_ids

        total_images = sum(len(patient_dict[pid]) for pid in patient_ids)
        target_val = int(total_images * val_frac)
        target_test = int(total_images * test_frac)

        val_ids, test_ids, train_ids = [], [], []
        running = 0
        for pid in patient_ids:
            n_imgs = len(patient_dict[pid])
            if running < target_test:
                test_ids.append(pid)
            elif running < target_test + target_val:
                val_ids.append(pid)
            else:
                train_ids.append(pid)
            running += n_imgs

        for pid in train_ids:
            train_fp += patient_dict[pid]; train_lb += [cls_idx] * len(patient_dict[pid])
        for pid in val_ids:
            val_fp += patient_dict[pid]; val_lb += [cls_idx] * len(patient_dict[pid])
        for pid in test_ids:
            test_fp += patient_dict[pid]; test_lb += [cls_idx] * len(patient_dict[pid])

        patient_counts["train"] += len(train_ids)
        patient_counts["val"] += len(val_ids)
        patient_counts["test"] += len(test_ids)

    print(f"Split sizes (images) -> train: {len(train_fp)}, val: {len(val_fp)}, test: {len(test_fp)}")
    print(f"Split sizes (patients) -> train: {patient_counts['train']}, "
          f"val: {patient_counts['val']}, test: {patient_counts['test']}")

    assert_no_patient_leakage(train_fp, val_fp, test_fp)

    return (train_fp, train_lb), (val_fp, val_lb), (test_fp, test_lb)


def assert_no_patient_leakage(train_fp, val_fp, test_fp):
    """Hard check: raises if any patient ID appears in more than one split.
    Run this after ANY split you intend to report numbers from — it's cheap
    and it's exactly the check a careful reviewer will ask whether you did."""
    train_ids = {extract_patient_id(fp) for fp in train_fp}
    val_ids = {extract_patient_id(fp) for fp in val_fp}
    test_ids = {extract_patient_id(fp) for fp in test_fp}

    overlap_train_val = train_ids & val_ids
    overlap_train_test = train_ids & test_ids
    overlap_val_test = val_ids & test_ids

    if overlap_train_val or overlap_train_test or overlap_val_test:
        raise ValueError(
            f"Patient leakage detected! "
            f"train/val overlap: {len(overlap_train_val)} patients, "
            f"train/test overlap: {len(overlap_train_test)} patients, "
            f"val/test overlap: {len(overlap_val_test)} patients."
        )
    print("Patient-leakage check passed: no patient appears in more than one split.")


def stratified_split(filepaths, labels, val_frac=0.10, test_frac=0.10, seed=42,
                      max_per_class=None):
    """DEPRECATED for reporting numbers — splits by IMAGE, not patient, so the
    same patient's B-scans can land in both train and test (leakage inflates
    test accuracy). Kept only for fast, throwaway smoke-testing of the
    pipeline itself. Use `patient_grouped_stratified_split` for anything you
    intend to put in a table or a paper.
    """
    rng = random.Random(seed)
    by_class = {i: [] for i in range(len(CLASSES))}
    for fp, lb in zip(filepaths, labels):
        by_class[lb].append(fp)

    train_fp, train_lb = [], []
    val_fp, val_lb = [], []
    test_fp, test_lb = [], []

    for cls_idx, files in by_class.items():
        rng.shuffle(files)
        if max_per_class is not None:
            files = files[:max_per_class]
        n = len(files)
        n_val = int(n * val_frac)
        n_test = int(n * test_frac)
        val_files = files[:n_val]
        test_files = files[n_val:n_val + n_test]
        train_files = files[n_val + n_test:]

        train_fp += train_files; train_lb += [cls_idx] * len(train_files)
        val_fp += val_files;     val_lb += [cls_idx] * len(val_files)
        test_fp += test_files;   test_lb += [cls_idx] * len(test_files)

    print(f"Split sizes -> train: {len(train_fp)}, val: {len(val_fp)}, test: {len(test_fp)}")
    print("NOTE: this is an image-level split (not patient-grouped) — do not "
          "report numbers from this split in a paper/thesis. Use "
          "patient_grouped_stratified_split instead.")
    return (train_fp, train_lb), (val_fp, val_lb), (test_fp, test_lb)


In [ ]:
%%writefile trustoct/model.py
"""
trustoct.model
ResNet50 (+ optional MSF, + optional CBAM) classifier for OCT.

IMPORTANT NAMING NOTE (read this before writing the paper):
TrustOCT is the FRAMEWORK — the evaluation methodology spanning metrics,
calibration, explainability faithfulness, and robustness (see trustoct/__init__.py).
The class below, `ResNetMSFCBAM`, is just ONE reference model used to
demonstrate that framework. Keeping these conceptually separate is what makes
the contribution "a framework" rather than "a CNN with a fancy name" — say so
explicitly in the paper's contribution statement.

Three experiment configs share this single class, controlled by two flags,
so the ablation (EXP001 -> EXP002 -> EXP003) is a true controlled comparison
(same backbone, same head, same training recipe):
    EXP001: use_msf=False, use_cbam=False   (plain ResNet50 baseline)
    EXP002: use_msf=True,  use_cbam=False   (+MSF)
    EXP003: use_msf=True,  use_cbam=True    (+MSF+CBAM)  <- the reference model

MSF here is used as a fusion mechanism to demonstrate the framework, not
presented as a novel architectural contribution in its own right — the paper's
novelty claim should rest on the evaluation methodology, not the backbone.
"""
import torch
import torch.nn as nn
import torchvision.models as tv_models

from trustoct.modules import CBAM, MSFModule


class ResNetMSFCBAM(nn.Module):
    def __init__(self, num_classes=4, use_msf=True, use_cbam=True,
                 pretrained=True, msf_out_channels=256, cbam_reduction=16):
        super().__init__()
        self.use_msf = use_msf
        self.use_cbam = use_cbam

        weights = tv_models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        backbone = tv_models.resnet50(weights=weights)

        # Split backbone into stages so we can tap intermediate feature maps.
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1   # 256 ch
        self.layer2 = backbone.layer2   # 512 ch
        self.layer3 = backbone.layer3   # 1024 ch
        self.layer4 = backbone.layer4   # 2048 ch

        if self.use_msf:
            self.msf = MSFModule(in_channels_list=[512, 1024, 2048], out_channels=msf_out_channels)
            head_in_channels = msf_out_channels
        else:
            head_in_channels = 2048

        if self.use_cbam:
            self.cbam = CBAM(head_in_channels, reduction_ratio=cbam_reduction)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(head_in_channels, num_classes)

        # Keep a handle to the last conv feature map for Grad-CAM/LayerCAM hooks.
        self._last_features = None

    def forward(self, x, return_features=False):
        x = self.stem(x)
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)

        if self.use_msf:
            feat = self.msf([c2, c3, c4])
        else:
            feat = c4

        if self.use_cbam:
            feat = self.cbam(feat)

        self._last_features = feat  # used by explainability hooks

        pooled = self.gap(feat).flatten(1)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        if return_features:
            return logits, feat
        return logits

    def get_target_layer(self):
        """Returns the module whose output activations/gradients LayerCAM should use.
        This is the final fused+attended feature map -> most semantically meaningful
        for a class-discriminative heatmap."""
        if self.use_cbam:
            return self.cbam
        if self.use_msf:
            return self.msf
        return self.layer4


EXPERIMENTS = {
    "EXP001_baseline_resnet50": dict(use_msf=False, use_cbam=False),
    "EXP002_resnet50_msf": dict(use_msf=True, use_cbam=False),
    "EXP003_resnet50_msf_cbam": dict(use_msf=True, use_cbam=True),
}


def build_model(exp_name, num_classes=4, pretrained=True):
    """Generic factory, kept for programmatic/looped use (e.g. multiseed.py)."""
    assert exp_name in EXPERIMENTS, f"Unknown exp_name. Choose from {list(EXPERIMENTS.keys())}"
    cfg = EXPERIMENTS[exp_name]
    return ResNetMSFCBAM(num_classes=num_classes, pretrained=pretrained, **cfg)


# --- Explicit named factories -------------------------------------------------
# Reviewer feedback: prefer explicit constructors over scattering boolean flags
# through calling code. Use these directly in notebook/paper code listings;
# build_model() above stays available for anywhere you need to loop over
# EXPERIMENTS programmatically (e.g. the multiseed runner).

def build_resnet50(num_classes=4, pretrained=True):
    """EXP001: plain ResNet50 baseline, no MSF, no CBAM."""
    return ResNetMSFCBAM(num_classes=num_classes, use_msf=False, use_cbam=False, pretrained=pretrained)


def build_resnet50_msf(num_classes=4, pretrained=True):
    """EXP002: ResNet50 + Multi-Scale Feature Fusion."""
    return ResNetMSFCBAM(num_classes=num_classes, use_msf=True, use_cbam=False, pretrained=pretrained)


def build_resnet50_msf_cbam(num_classes=4, pretrained=True):
    """EXP003: ResNet50 + MSF + CBAM — the TrustOCT reference model."""
    return ResNetMSFCBAM(num_classes=num_classes, use_msf=True, use_cbam=True, pretrained=pretrained)


# Backward-compatible alias — remove before final submission once all
# notebook cells / saved-script references are confirmed migrated to the new name.
TrustOCTNet = ResNetMSFCBAM


In [ ]:
%%writefile trustoct/train.py
"""
trustoct.train
Training loop shared by all three experiments (EXP001/002/003), so the only
difference between runs is the model architecture — everything else (optimizer,
schedule, augmentation, loss, seed) is held fixed for a fair ablation.
"""
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from trustoct.utils import AverageMeter, save_checkpoint, get_device


def compute_class_weights(labels, num_classes):
    """Inverse-frequency class weights, used in the loss to counter Kermany's
    class imbalance (NORMAL is undersampled relative to CNV/DME/DRUSEN)."""
    counts = torch.zeros(num_classes)
    for lb in labels:
        counts[lb] += 1
    weights = counts.sum() / (num_classes * counts.clamp(min=1))
    return weights


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_meter, acc_meter = AverageMeter(), AverageMeter()
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)
        acc = (preds == labels).float().mean().item()
        loss_meter.update(loss.item(), images.size(0))
        acc_meter.update(acc, images.size(0))
    return loss_meter.avg, acc_meter.avg


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    loss_meter, acc_meter = AverageMeter(), AverageMeter()
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        preds = logits.argmax(dim=1)
        acc = (preds == labels).float().mean().item()
        loss_meter.update(loss.item(), images.size(0))
        acc_meter.update(acc, images.size(0))
    return loss_meter.avg, acc_meter.avg


def fit(model, train_ds, val_ds, exp_name, epochs=25, batch_size=32, lr=1e-4,
        weight_decay=1e-4, num_workers=2, patience=5, ckpt_dir="/content/checkpoints",
        class_weights=None, overfit_gap_threshold=0.15, overfit_patience=3,
        min_epochs=5, verbose=True):
    """Full training run with TWO independent early-stopping triggers, plus
    best-checkpoint saving. Returns the trained model (best weights loaded) and
    a history dict (including a per-epoch overfit_gap) for the loss/accuracy
    curves you'll want in the thesis.

    Early-stop triggers (either one alone can end training):

    1. **Val-loss plateau** (`patience`): stops if val_loss hasn't improved for
       `patience` consecutive epochs. Catches the case where the model has
       simply stopped getting better.

    2. **Overfitting gap** (`overfit_gap_threshold`, `overfit_patience`): stops
       if `train_acc - val_acc` exceeds `overfit_gap_threshold` for
       `overfit_patience` consecutive epochs. This catches the case a pure
       val-loss patience check MISSES — val_loss can still be (slowly)
       improving even while the train/val gap widens, especially with a
       pretrained backbone that memorizes quickly. `min_epochs` guards against
       triggering this before the model has had a chance to warm up.

    In both cases the model reverts to the BEST checkpoint by val_loss (not the
    epoch training stopped at), so an overfitting-triggered stop still returns
    the best generalizing weights seen so far, not an already-overfit model.
    """
    device = get_device()
    model = model.to(device)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device) if class_weights is not None else None)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    epochs_no_improve = 0
    overfit_streak = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "overfit_gap": []}
    stop_reason = "completed all epochs"

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        gap = tr_acc - val_acc  # positive & growing -> overfitting signature

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["overfit_gap"].append(gap)

        elapsed = time.time() - t0
        if verbose:
            flag = "  <- overfit gap high" if gap > overfit_gap_threshold else ""
            print(f"[{exp_name}] epoch {epoch:02d}/{epochs} | "
                  f"train_loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val_loss {val_loss:.4f} acc {val_acc:.4f} | "
                  f"gap {gap:+.4f} | {elapsed:.1f}s{flag}")

        # --- checkpoint on best val_loss ---
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_no_improve = 0
            save_checkpoint(model, optimizer, epoch, best_val_loss,
                             f"{ckpt_dir}/{exp_name}_best.pt")
        else:
            epochs_no_improve += 1

        # --- overfitting-gap trigger ---
        if epoch >= min_epochs and gap > overfit_gap_threshold:
            overfit_streak += 1
        else:
            overfit_streak = 0

        # --- check both stopping conditions ---
        if epochs_no_improve >= patience:
            stop_reason = (f"val_loss plateaued for {patience} epochs "
                            f"(best val_loss={best_val_loss:.4f} at epoch {best_epoch})")
            break
        if overfit_streak >= overfit_patience:
            stop_reason = (f"train/val accuracy gap exceeded {overfit_gap_threshold:.2f} "
                            f"for {overfit_patience} consecutive epochs "
                            f"(gap={gap:.3f} at epoch {epoch}) — reverting to best "
                            f"checkpoint from epoch {best_epoch}")
            break

    if verbose:
        print(f"[{exp_name}] Stopped: {stop_reason}")
        print(f"[{exp_name}] Restoring best weights from epoch {best_epoch} "
              f"(val_loss={best_val_loss:.4f}).")

    model.load_state_dict(best_state)
    history["stop_reason"] = stop_reason
    history["best_epoch"] = best_epoch
    return model, history


In [ ]:
%%writefile trustoct/metrics.py
"""
trustoct.metrics
Standard metrics table for the ablation (EXP001 vs EXP002 vs EXP003) — this is the
single most-referenced table in the paper, so get its computation airtight and
identical across all three experiments.
"""
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score,
    roc_auc_score, confusion_matrix, classification_report,
)


@torch.no_grad()
def get_predictions(model, loader, device):
    """Runs the model over a loader once, returns (y_true, y_pred, y_prob, paths)."""
    model.eval()
    all_labels, all_preds, all_probs, all_paths = [], [], [], []
    for images, labels, paths in loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_labels.append(labels.numpy())
        all_preds.append(preds)
        all_probs.append(probs)
        all_paths.extend(paths)
    return (np.concatenate(all_labels), np.concatenate(all_preds),
            np.concatenate(all_probs), all_paths)


def specificity_per_class(y_true, y_pred, num_classes):
    """Specificity isn't in sklearn directly — derive it from the confusion matrix
    per class (TN / (TN+FP)), then macro-average."""
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    specs = []
    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return float(np.mean(specs)), specs


def compute_metrics(y_true, y_pred, y_prob, num_classes=4, class_names=None):
    """Returns a flat dict of the metrics you'll put in the ablation table:
    accuracy, macro precision/recall/F1, balanced accuracy, specificity, MCC,
    Cohen's kappa, macro (one-vs-rest) ROC-AUC."""
    macro_spec, per_class_spec = specificity_per_class(y_true, y_pred, num_classes)

    try:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")  # can happen if a class is absent from a small eval split

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "specificity_macro": macro_spec,
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "roc_auc_macro": auc,
    }
    return metrics


def build_ablation_table(results_dict, num_classes=4):
    """results_dict: {exp_name: (y_true, y_pred, y_prob)} -> pandas DataFrame with
    one row per experiment, ready to paste into the paper (Table 2 in most OCT
    ablation papers)."""
    rows = []
    for exp_name, (y_true, y_pred, y_prob) in results_dict.items():
        m = compute_metrics(y_true, y_pred, y_prob, num_classes)
        m["experiment"] = exp_name
        rows.append(m)
    df = pd.DataFrame(rows).set_index("experiment")
    col_order = ["accuracy", "precision_macro", "recall_macro", "specificity_macro",
                 "f1_macro", "balanced_accuracy", "mcc", "cohen_kappa", "roc_auc_macro"]
    return df[col_order].round(4)


def print_classwise_report(y_true, y_pred, class_names):
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))


In [ ]:
%%writefile trustoct/calibration.py
"""
trustoct.calibration
Phase 3 core differentiator, part A: is the model's confidence trustworthy?
Most OCT papers report accuracy only; a model can be accurate yet badly
overconfident, which matters clinically (a 99%-confident wrong prediction is more
dangerous than a 55%-confident wrong one). We quantify this with:
  - Expected Calibration Error (ECE)
  - Brier score (multiclass)
  - Reliability diagrams (plotted, for EXP001 vs EXP003)
"""
import numpy as np
import matplotlib.pyplot as plt


def expected_calibration_error(y_true, y_prob, n_bins=15):
    """ECE = sum over bins of (|bin|/N) * |accuracy(bin) - confidence(bin)|,
    using max predicted probability (top-1 confidence) per sample, standard
    definition from Guo et al. 2017 ('On Calibration of Modern Neural Networks')."""
    confidences = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_stats = []
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        in_bin = (confidences > lo) & (confidences <= hi) if i > 0 else \
                 (confidences >= lo) & (confidences <= hi)
        prop_in_bin = in_bin.mean()
        if prop_in_bin > 0:
            acc_in_bin = accuracies[in_bin].mean()
            conf_in_bin = confidences[in_bin].mean()
            ece += np.abs(acc_in_bin - conf_in_bin) * prop_in_bin
            bin_stats.append((lo, hi, acc_in_bin, conf_in_bin, prop_in_bin))
        else:
            bin_stats.append((lo, hi, np.nan, np.nan, 0.0))
    return float(ece), bin_stats


def brier_score_multiclass(y_true, y_prob, num_classes):
    """Multiclass Brier score: mean squared error between predicted probability
    vector and one-hot true label, averaged over samples. Lower is better; a
    perfectly calibrated + accurate model scores 0."""
    y_onehot = np.eye(num_classes)[y_true]
    return float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))


def plot_reliability_diagram(bin_stats_dict, title="Reliability Diagram", save_path=None):
    """bin_stats_dict: {exp_name: bin_stats} where bin_stats comes from
    expected_calibration_error(). Plots one line per experiment against the
    perfect-calibration diagonal — this is the figure that visually backs up
    your ECE numbers in the paper."""
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")

    for exp_name, bin_stats in bin_stats_dict.items():
        confs = [b[3] for b in bin_stats if not np.isnan(b[3])]
        accs = [b[2] for b in bin_stats if not np.isnan(b[2])]
        ax.plot(confs, accs, marker="o", label=exp_name)

    ax.set_xlabel("Mean predicted confidence")
    ax.set_ylabel("Empirical accuracy")
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200)
    return fig


def calibration_report(y_true, y_prob, num_classes, n_bins=15):
    """
    Beyond ECE/Brier, also reports average confidence and average correctness
    (accuracy) as plain scalars — simple numbers, but they let a reader
    interpret the DIRECTION of miscalibration at a glance: if avg_confidence >
    avg_accuracy the model is overconfident (the more clinically concerning
    direction); if avg_confidence < avg_accuracy it's underconfident.
    """
    ece, bin_stats = expected_calibration_error(y_true, y_prob, n_bins)
    brier = brier_score_multiclass(y_true, y_prob, num_classes)

    confidences = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    avg_confidence = float(confidences.mean())
    avg_accuracy = float(accuracies.mean())

    return {
        "ece": ece,
        "brier_score": brier,
        "avg_confidence": avg_confidence,
        "avg_accuracy": avg_accuracy,
        "confidence_minus_accuracy": avg_confidence - avg_accuracy,  # >0 = overconfident
    }, bin_stats


In [ ]:
%%writefile trustoct/explainability.py
"""
trustoct.explainability
Phase 3 core differentiator, part B: are the model's explanations faithful, not
just pretty? Most OCT papers stop at a qualitative Grad-CAM picture. We add:
  - LayerCAM (Jiang et al. 2021) — finer-grained than Grad-CAM since it uses
    positive per-pixel gradient*activation rather than a single global-average
    weight per channel, which matters for small OCT lesions (e.g. early drusen).
  - Deletion/Insertion AOPC (Area Over Perturbation Curve, Samek et al. 2017) —
    a *quantitative* faithfulness score: does removing the pixels LayerCAM marks
    "important" actually hurt the prediction? This is the number that turns
    "look, a heatmap" into a scientific claim.
"""
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.cm as cm


class LayerCAM:
    """Hooks a target layer's forward activations and backward gradients, then
    computes: CAM = ReLU(sum_k ReLU(grad_k) * activation_k), upsampled to input
    resolution. Call `generate(image_tensor, class_idx)` for a single image.
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, inp, out):
            self.activations = out.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, image_tensor, class_idx=None):
        """image_tensor: [1, C, H, W], already normalized, on the correct device.
        Returns (cam [H, W] in [0,1], predicted_class_idx, predicted_prob)."""
        self.model.eval()
        image_tensor = image_tensor.clone().requires_grad_(True)
        logits = self.model(image_tensor)
        probs = torch.softmax(logits, dim=1)

        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())

        self.model.zero_grad()
        score = logits[0, class_idx]
        score.backward()

        # LayerCAM weighting: positive gradients only, elementwise (not GAP-pooled
        # like Grad-CAM), which preserves finer spatial detail.
        weights = F.relu(self.gradients)
        weighted_activations = weights * self.activations
        cam = F.relu(weighted_activations.sum(dim=1, keepdim=True))  # [1,1,h,w]

        cam = F.interpolate(cam, size=image_tensor.shape[-2:], mode="bilinear",
                             align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        return cam, class_idx, float(probs[0, class_idx].item())


def overlay_cam_on_image(image_np, cam, alpha=0.45, colormap="jet"):
    """image_np: [H, W, 3] float in [0,1]. cam: [H, W] float in [0,1].
    Returns an RGB overlay for the figure in your paper."""
    heatmap = cm.get_cmap(colormap)(cam)[:, :, :3]
    overlay = (1 - alpha) * image_np + alpha * heatmap
    return np.clip(overlay, 0, 1)


# ---------------------------------------------------------------------------
# Deletion / Insertion AOPC — quantitative faithfulness
# ---------------------------------------------------------------------------
@torch.no_grad()
def _predict_prob(model, image_tensor, class_idx):
    logits = model(image_tensor)
    prob = torch.softmax(logits, dim=1)[0, class_idx].item()
    return prob


def deletion_insertion_curves(model, image_tensor, cam, class_idx, device,
                               num_steps=20, baseline_value=0.0):
    """Ranks pixels by CAM importance (descending), then:
      - Deletion: progressively replaces the *most* important pixels with
        `baseline_value` (mean-normalized black) and re-scores -> a faithful CAM
        should cause a STEEP drop (low AOPC = good deletion behavior, i.e. area
        UNDER the curve is small).
      - Insertion: starts from a blank image and progressively reveals the *most*
        important pixels first -> a faithful CAM should cause a STEEP rise
        (high AOPC = good insertion behavior).
    Returns (deletion_scores, insertion_scores, del_aopc, ins_aopc) where AOPC is
    computed as area-over/under the respective curve using the trapezoidal rule.
    """
    model.eval()
    img = image_tensor.clone().to(device)  # [1, C, H, W]
    C, H, W = img.shape[1], img.shape[2], img.shape[3]

    flat_order = np.argsort(-cam.flatten())  # descending importance
    total_pixels = H * W
    step_size = max(total_pixels // num_steps, 1)

    # --- Deletion ---
    del_img = img.clone()
    deletion_scores = [_predict_prob(model, del_img, class_idx)]
    mask_flat = np.ones(total_pixels, dtype=bool)
    for step in range(1, num_steps + 1):
        idx_to_remove = flat_order[(step - 1) * step_size: step * step_size]
        mask_flat[idx_to_remove] = False
        mask_2d = torch.tensor(mask_flat.reshape(H, W), device=device, dtype=torch.float32)
        del_img = image_tensor.to(device) * mask_2d + baseline_value * (1 - mask_2d)
        deletion_scores.append(_predict_prob(model, del_img, class_idx))

    # --- Insertion ---
    ins_img_base = torch.full_like(img, baseline_value)
    insertion_scores = [_predict_prob(model, ins_img_base, class_idx)]
    mask_flat = np.zeros(total_pixels, dtype=bool)
    for step in range(1, num_steps + 1):
        idx_to_add = flat_order[(step - 1) * step_size: step * step_size]
        mask_flat[idx_to_add] = True
        mask_2d = torch.tensor(mask_flat.reshape(H, W), device=device, dtype=torch.float32)
        ins_img = image_tensor.to(device) * mask_2d + baseline_value * (1 - mask_2d)
        insertion_scores.append(_predict_prob(model, ins_img, class_idx))

    _trapz = getattr(np, "trapezoid", None) or np.trapz  # numpy>=2.0 renamed trapz
    x_axis = np.linspace(0, 1, num_steps + 1)
    del_aopc = float(_trapz(deletion_scores, x_axis))   # want this LOW
    ins_aopc = float(_trapz(insertion_scores, x_axis))  # want this HIGH

    return deletion_scores, insertion_scores, del_aopc, ins_aopc


def faithfulness_report(model, cam_engine, loader, device, num_samples=50, num_steps=20):
    """Runs deletion/insertion AOPC over a random subset of the test set (full test
    set is expensive: num_samples x num_steps x 2 forward passes each). Returns
    mean deletion AOPC (lower=better) and mean insertion AOPC (higher=better) —
    the two numbers that go in your Phase-3 table next to the ECE/Brier numbers."""
    del_aopcs, ins_aopcs = [], []
    seen = 0
    for images, labels, _ in loader:
        for i in range(images.size(0)):
            if seen >= num_samples:
                break
            img = images[i:i + 1].to(device)
            cam, pred_idx, _ = cam_engine.generate(img)
            _, _, del_aopc, ins_aopc = deletion_insertion_curves(
                model, img, cam, pred_idx, device, num_steps=num_steps)
            del_aopcs.append(del_aopc)
            ins_aopcs.append(ins_aopc)
            seen += 1
        if seen >= num_samples:
            break
    return {
        "mean_deletion_aopc": float(np.mean(del_aopcs)),
        "mean_insertion_aopc": float(np.mean(ins_aopcs)),
        "n_samples": seen,
    }


def plot_cam_grid(images_np, cams, titles, save_path=None):
    """Small qualitative grid: original | heatmap | overlay, for 4-6 examples
    (your Phase-4 failure-analysis figure can reuse this too)."""
    n = len(images_np)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1:
        axes = axes[None, :]
    for i in range(n):
        axes[i, 0].imshow(images_np[i]); axes[i, 0].set_title(f"{titles[i]} - input"); axes[i, 0].axis("off")
        axes[i, 1].imshow(cams[i], cmap="jet"); axes[i, 1].set_title("LayerCAM"); axes[i, 1].axis("off")
        overlay = overlay_cam_on_image(images_np[i], cams[i])
        axes[i, 2].imshow(overlay); axes[i, 2].set_title("Overlay"); axes[i, 2].axis("off")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200)
    return fig


In [ ]:
%%writefile trustoct/robustness.py
"""
trustoct.robustness
Phase 4 (supporting, keep lean): does accuracy hold up under realistic OCT
acquisition noise? 4 perturbation types x a few severities, one summary table.
"""
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score


def add_gaussian_noise(img_tensor, severity):
    sigma = [0.02, 0.05, 0.08, 0.12][severity - 1]
    noise = torch.randn_like(img_tensor) * sigma
    return torch.clamp(img_tensor + noise, -3, 3)  # normalized-space clamp


def add_gaussian_blur(img_tensor, severity):
    ksize = [3, 5, 7, 9][severity - 1]
    sigma = ksize / 6.0
    channels = img_tensor.shape[1]
    coords = torch.arange(ksize, dtype=torch.float32) - ksize // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = (g / g.sum()).to(img_tensor.device)
    kernel_1d = g.view(1, 1, 1, ksize)
    kernel_1d_t = g.view(1, 1, ksize, 1)
    kernel_1d = kernel_1d.repeat(channels, 1, 1, 1)
    kernel_1d_t = kernel_1d_t.repeat(channels, 1, 1, 1)
    pad = ksize // 2
    x = F.conv2d(img_tensor, kernel_1d, padding=(0, pad), groups=channels)
    x = F.conv2d(x, kernel_1d_t, padding=(pad, 0), groups=channels)
    return x


def adjust_brightness(img_tensor, severity, mean, std):
    """Adjust brightness in *unnormalized* pixel space then re-normalize, so the
    perturbation magnitude is physically meaningful (fraction of pixel range)."""
    delta = [0.06, 0.12, 0.20, 0.30][severity - 1]
    mean_t = torch.tensor(mean, device=img_tensor.device).view(1, -1, 1, 1)
    std_t = torch.tensor(std, device=img_tensor.device).view(1, -1, 1, 1)
    pixel = img_tensor * std_t + mean_t
    pixel = torch.clamp(pixel + delta, 0, 1)
    return (pixel - mean_t) / std_t


def adjust_contrast(img_tensor, severity, mean, std):
    factor = [0.9, 0.75, 0.6, 0.45][severity - 1]
    mean_t = torch.tensor(mean, device=img_tensor.device).view(1, -1, 1, 1)
    std_t = torch.tensor(std, device=img_tensor.device).view(1, -1, 1, 1)
    pixel = img_tensor * std_t + mean_t
    gray_mean = pixel.mean(dim=[2, 3], keepdim=True)
    pixel = torch.clamp((pixel - gray_mean) * factor + gray_mean, 0, 1)
    return (pixel - mean_t) / std_t


PERTURBATIONS = {
    "gaussian_noise": add_gaussian_noise,
    "gaussian_blur": add_gaussian_blur,
}
NORM_SPACE_PERTURBATIONS = {
    "brightness": adjust_brightness,
    "contrast": adjust_contrast,
}


@torch.no_grad()
def evaluate_under_perturbation(model, loader, device, mean, std, severities=(1, 2, 3)):
    """Runs the test set through each perturbation type/severity and records
    accuracy + macro-F1 drop relative to clean performance. One row per
    (perturbation, severity) in the returned list -> becomes your Phase-4 table."""
    model.eval()
    results = []

    def run_pass(perturb_fn, name, needs_norm_stats):
        for sev in severities:
            all_labels, all_preds = [], []
            for images, labels, _ in loader:
                images = images.to(device)
                if needs_norm_stats:
                    images_p = perturb_fn(images, sev, mean, std)
                else:
                    images_p = perturb_fn(images, sev)
                logits = model(images_p)
                preds = logits.argmax(dim=1).cpu().numpy()
                all_preds.append(preds)
                all_labels.append(labels.numpy())
            y_true = np.concatenate(all_labels)
            y_pred = np.concatenate(all_preds)
            results.append({
                "perturbation": name,
                "severity": sev,
                "accuracy": accuracy_score(y_true, y_pred),
                "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            })

    for name, fn in PERTURBATIONS.items():
        run_pass(fn, name, needs_norm_stats=False)
    for name, fn in NORM_SPACE_PERTURBATIONS.items():
        run_pass(fn, name, needs_norm_stats=True)

    return results


In [ ]:
%%writefile trustoct/multiseed.py
"""
trustoct.multiseed
Statistical significance across the EXP001/002/003 ablation.

A single run per experiment invites the obvious reviewer question: "is that
accuracy difference meaningful, or noise?" This module trains each experiment
config across multiple random seeds and reports mean ± std for every metric,
plus a paired t-test / Wilcoxon signed-rank test between the reference model
(EXP003) and the baseline (EXP001) on per-seed metric values.

Usage (in the Colab notebook):

    from trustoct.multiseed import run_multiseed_ablation
    results = run_multiseed_ablation(
        train_ds_fn, val_ds_fn, test_ds_fn,   # callables: seed -> Dataset
        seeds=[42, 123, 2024],
        epochs=25, batch_size=32, lr=1e-4, ckpt_dir="/content/checkpoints",
    )

Note on cost: this multiplies total training time by len(seeds). With 3 seeds
across 3 experiments that's 9 full training runs — budget Colab GPU time
accordingly. If time is tight, 3 seeds is the minimum that's still defensible
in a viva/review ("we repeated with 3 random seeds"); fewer than that, don't
claim statistical significance at all — just report single-run numbers
honestly and note this as a limitation.
"""
import numpy as np
import pandas as pd
from scipy import stats

from trustoct.model import build_model, EXPERIMENTS
from trustoct.train import fit, compute_class_weights
from trustoct.metrics import get_predictions, compute_metrics
from trustoct.utils import set_seed, get_device


def run_single_seed(exp_name, train_ds, val_ds, test_ds, seed, num_classes=4,
                     epochs=25, batch_size=32, lr=1e-4, weight_decay=1e-4,
                     ckpt_dir="/content/checkpoints", patience=5, verbose=False):
    """Trains one (experiment, seed) combination end-to-end and returns test-set metrics."""
    set_seed(seed)
    device = get_device()

    model = build_model(exp_name, num_classes=num_classes)
    class_weights = compute_class_weights(train_ds.labels, num_classes)

    model, history = fit(
        model, train_ds, val_ds, exp_name=f"{exp_name}_seed{seed}",
        epochs=epochs, batch_size=batch_size, lr=lr, weight_decay=weight_decay,
        ckpt_dir=ckpt_dir, patience=patience, class_weights=class_weights, verbose=verbose,
    )

    from torch.utils.data import DataLoader
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    y_true, y_pred, y_prob, _ = get_predictions(model, test_loader, device)
    metrics = compute_metrics(y_true, y_pred, y_prob, num_classes=num_classes)
    metrics["seed"] = seed
    metrics["experiment"] = exp_name
    return metrics, model


def run_multiseed_ablation(train_ds, val_ds, test_ds, seeds=(42, 123, 2024),
                            num_classes=4, epochs=25, batch_size=32, lr=1e-4,
                            weight_decay=1e-4, ckpt_dir="/content/checkpoints",
                            patience=5, verbose=True):
    """
    Runs all three EXPERIMENTS across all seeds. Datasets are fixed across
    seeds here (only model init + training stochasticity varies) — that's the
    correct design for "does the architecture reliably help," as opposed to
    also varying the data split, which would conflate two different sources
    of variance.

    Returns:
        per_run_df: one row per (experiment, seed) — the raw data.
        summary_df: mean ± std per experiment, ready for the paper's table.
        significance: dict of paired-test results, EXP003 vs EXP001 and
                      EXP003 vs EXP002, per metric.
    """
    all_rows = []
    for exp_name in EXPERIMENTS:
        for seed in seeds:
            if verbose:
                print(f"\n=== Running {exp_name} | seed={seed} ===")
            metrics, _ = run_single_seed(
                exp_name, train_ds, val_ds, test_ds, seed,
                num_classes=num_classes, epochs=epochs, batch_size=batch_size,
                lr=lr, weight_decay=weight_decay, ckpt_dir=ckpt_dir,
                patience=patience, verbose=verbose,
            )
            all_rows.append(metrics)

    per_run_df = pd.DataFrame(all_rows)

    metric_cols = [c for c in per_run_df.columns if c not in ("seed", "experiment")]
    summary_rows = []
    for exp_name in EXPERIMENTS:
        sub = per_run_df[per_run_df["experiment"] == exp_name]
        row = {"experiment": exp_name}
        for m in metric_cols:
            row[f"{m}_mean"] = sub[m].mean()
            row[f"{m}_std"] = sub[m].std(ddof=1) if len(sub) > 1 else 0.0
        summary_rows.append(row)
    summary_df = pd.DataFrame(summary_rows).set_index("experiment")

    significance = {}
    exp_names = list(EXPERIMENTS.keys())
    reference = exp_names[-1]  # EXP003
    for baseline in exp_names[:-1]:
        sig_for_baseline = {}
        ref_sub = per_run_df[per_run_df["experiment"] == reference].sort_values("seed")
        base_sub = per_run_df[per_run_df["experiment"] == baseline].sort_values("seed")
        for m in metric_cols:
            if len(ref_sub) >= 2 and len(base_sub) >= 2 and len(ref_sub) == len(base_sub):
                t_stat, p_val = stats.ttest_rel(ref_sub[m].values, base_sub[m].values)
                sig_for_baseline[m] = {"t_stat": float(t_stat), "p_value": float(p_val)}
            else:
                sig_for_baseline[m] = {
                    "t_stat": float("nan"), "p_value": float("nan"),
                    "note": "need >=2 matched seeds per experiment for a paired test",
                }
        significance[f"{reference}_vs_{baseline}"] = sig_for_baseline

    return per_run_df, summary_df, significance


def format_mean_std_table(summary_df, metrics_to_show=None):
    """Formats summary_df into 'mean ± std' strings for direct paste into the paper."""
    if metrics_to_show is None:
        metrics_to_show = ["accuracy", "f1_macro", "roc_auc_macro", "mcc"]
    out = pd.DataFrame(index=summary_df.index)
    for m in metrics_to_show:
        out[m] = summary_df.apply(lambda r: f"{r[f'{m}_mean']:.4f} ± {r[f'{m}_std']:.4f}", axis=1)
    return out


In [ ]:
%%writefile trustoct/external_validation.py
"""
trustoct.external_validation
Tests the trained reference model's generalization on an INDEPENDENT OCT
dataset it never saw during training or model selection.

Why this matters for the paper: 96–97% accuracy on Kermany's own test split is
expected (it's a large, relatively clean, single-source dataset) and reviewers
increasingly ask "does this hold up on a different scanner/population?" without
it, "trustworthy" only covers in-distribution behavior. Even a modest external
set is stronger evidence than none.

Suggested independent datasets (pick one you can access):
  - OCTID (Optical Coherence Tomography Image Database) — public, different
    acquisition source from Kermany.
  - Duke OCT datasets (Srinivasan et al. / Farsiu Duke AMD dataset) — different
    patient population and device.
  - Any local hospital/clinic OCT scans you have ethical clearance to use.

This module makes NO assumption about label schema matching exactly — you'll
likely need a small mapping step (external_label -> {CNV, DME, DRUSEN, NORMAL})
since class taxonomies differ slightly across public OCT datasets. Do that
mapping explicitly and document it in the paper's dataset section; don't
silently drop mismatched classes without saying so.
"""
import os
import torch
from torch.utils.data import DataLoader

from trustoct.data import OCTDataset, build_transforms, CLASSES
from trustoct.metrics import get_predictions, compute_metrics, print_classwise_report
from trustoct.calibration import calibration_report
from trustoct.utils import get_device


def index_external_folder(root_dir, class_folder_map):
    """
    root_dir: path to the external dataset root, expected to contain one
              subfolder per class (folder names may differ from Kermany's).
    class_folder_map: dict mapping external folder name -> Kermany-schema
              class name, e.g. {"AMD": "DRUSEN", "Normal": "NORMAL", ...}.
              Any external class NOT in this map is skipped (and reported),
              rather than silently mis-assigned.
    """
    filepaths, labels, skipped = [], [], []
    for folder_name in sorted(os.listdir(root_dir)):
        folder_path = os.path.join(root_dir, folder_name)
        if not os.path.isdir(folder_path):
            continue
        if folder_name not in class_folder_map:
            skipped.append(folder_name)
            continue
        mapped_class = class_folder_map[folder_name]
        class_idx = CLASSES.index(mapped_class)
        for fname in os.listdir(folder_path):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                filepaths.append(os.path.join(folder_path, fname))
                labels.append(class_idx)

    if skipped:
        print(f"NOTE: skipped external folders with no class mapping: {skipped}. "
              f"Add them to class_folder_map if they should be included.")
    print(f"External validation set indexed: {len(filepaths)} images across "
          f"{len(set(labels))} mapped classes.")
    return filepaths, labels


def run_external_validation(model, root_dir, class_folder_map, image_size=224,
                             batch_size=32, num_classes=4, save_calibration_plot=None):
    """
    Evaluates an already-trained model (e.g. your best EXP003 checkpoint) on
    the external dataset. Returns metrics + calibration report, computed
    identically to the in-distribution test evaluation so the two are directly
    comparable in the paper (put them side by side in one table — that
    comparison IS the generalization result).
    """
    filepaths, labels = index_external_folder(root_dir, class_folder_map)
    if len(filepaths) == 0:
        raise RuntimeError("No external images matched the provided class_folder_map — check paths/mapping.")

    transform = build_transforms(image_size=image_size, train=False, use_clahe=True)
    ext_ds = OCTDataset(filepaths, labels, transform=transform)
    ext_loader = DataLoader(ext_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    device = get_device()
    model = model.to(device)
    y_true, y_pred, y_prob, paths = get_predictions(model, ext_loader, device)

    metrics = compute_metrics(y_true, y_pred, y_prob, num_classes=num_classes)
    cal_report, bin_stats = calibration_report(y_true, y_prob, num_classes=num_classes)

    print("=== External Validation Results ===")
    print_classwise_report(y_true, y_pred, CLASSES)
    print(f"Accuracy: {metrics['accuracy']:.4f} | F1 (macro): {metrics['f1_macro']:.4f} | "
          f"ROC-AUC (macro): {metrics['roc_auc_macro']:.4f}")
    print(f"ECE: {cal_report['ece']:.4f} | Brier: {cal_report['brier_score']:.4f} | "
          f"avg confidence: {cal_report['avg_confidence']:.4f} vs avg accuracy: {cal_report['avg_accuracy']:.4f}")

    if save_calibration_plot:
        from trustoct.calibration import plot_reliability_diagram
        plot_reliability_diagram({"external_validation": bin_stats},
                                  title="Reliability — External Validation Set",
                                  save_path=save_calibration_plot)

    return {
        "metrics": metrics,
        "calibration": cal_report,
        "y_true": y_true, "y_pred": y_pred, "y_prob": y_prob, "paths": paths,
    }


In [ ]:
# Reload if this cell block was re-run after edits
import importlib, sys
for m in list(sys.modules):
    if m.startswith("trustoct"):
        del sys.modules[m]

from trustoct import *
set_seed(42)
device = get_device()
print("Using device:", device)


## 3. Download and prepare the Kermany OCT2017 dataset
Kaggle dataset: `paultimothymooney/kermany2018`. We use only the **train** folder
(~84,000 images) and re-split it ourselves 80/10/10, because Kermany's own
`test` folder only has 8 images/class — far too small for a stable test-set
metric to report in a paper.

**Patient-level split, not image-level.** Kermany filenames encode a patient ID
(e.g. `CNV-1016042-1.jpeg`), and a single patient contributes multiple B-scans
that are highly correlated with each other. Splitting by *image* lets the same
patient appear in both train and test — inflating reported accuracy in a way
that doesn't reflect real generalization. This notebook uses
`patient_grouped_stratified_split`, which keeps every patient's images
entirely within one split, and calls `assert_no_patient_leakage` to verify it
— this check (and passing it) is something to explicitly report in your
methodology section, since it's increasingly something reviewers check for on
this exact dataset.

Set `MAX_PER_CLASS` to cap dataset size for faster experimentation on Colab's free
GPU tier (e.g. `2000`); set to `None` to use the full dataset for your final numbers.


In [ ]:
MAX_PER_CLASS = 4000   # set to None for the full ~84k-image dataset (final paper run)

dataset_path = download_kermany_dataset()
# kagglehub typically returns .../versions/N ; the OCT2017 train folder sits under OCT2017/train
import glob
candidates = glob.glob(os.path.join(dataset_path, "**", "OCT2017*", "train"), recursive=True)
assert len(candidates) > 0, f"Could not locate OCT2017/train under {dataset_path} - inspect the folder manually."
TRAIN_DIR = candidates[0]
print("Using train dir:", TRAIN_DIR)


In [ ]:
filepaths, labels = index_kermany_folder(TRAIN_DIR)
print(f"Total images found: {len(filepaths)}")
for i, c in enumerate(CLASSES):
    print(f"  {c}: {labels.count(i)}")

(train_fp, train_lb), (val_fp, val_lb), (test_fp, test_lb) = patient_grouped_stratified_split(
    filepaths, labels, val_frac=0.10, test_frac=0.10, seed=42, max_per_class=MAX_PER_CLASS
)
# Belt-and-braces: patient_grouped_stratified_split already asserts this internally,
# but re-checking explicitly here makes the guarantee visible in the notebook output
# you'd screenshot for a thesis appendix.
assert_no_patient_leakage(train_fp, val_fp, test_fp)


In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

train_tfm = build_transforms(IMAGE_SIZE, train=True, use_clahe=True)
eval_tfm = build_transforms(IMAGE_SIZE, train=False, use_clahe=True)

train_ds = OCTDataset(train_fp, train_lb, transform=train_tfm)
val_ds = OCTDataset(val_fp, val_lb, transform=eval_tfm)
test_ds = OCTDataset(test_fp, test_lb, transform=eval_tfm)

class_weights = compute_class_weights(train_lb, num_classes=len(CLASSES))
print("Class weights (inverse-frequency):", class_weights)


In [ ]:
# Quick visual sanity check: a few CLAHE-preprocessed training images
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    img, label, path = train_ds[i * 200]
    denorm = img.permute(1, 2, 0).numpy() * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    ax.imshow(denorm.clip(0, 1))
    ax.set_title(CLASSES[label])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Phase 1 — Baseline (EXP001: plain ResNet50)
Establishes the number every later architectural change is compared against.


In [ ]:
from torch.utils.data import DataLoader

EPOCHS = 25          # reduce to ~8-10 for a fast smoke test, use full 25 for final numbers
LR = 1e-4
PATIENCE = 5              # stop if val_loss hasn't improved for this many epochs
OVERFIT_GAP_THRESHOLD = 0.15   # stop if (train_acc - val_acc) exceeds this...
OVERFIT_PATIENCE = 3           # ...for this many consecutive epochs
MIN_EPOCHS = 5                 # don't let the overfit trigger fire before this epoch (warm-up)

model_exp001 = build_model("EXP001_baseline_resnet50", num_classes=len(CLASSES), pretrained=True)
print(count_parameters(model_exp001))

model_exp001, history_exp001 = fit(
    model_exp001, train_ds, val_ds, exp_name="EXP001_baseline_resnet50",
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    overfit_gap_threshold=OVERFIT_GAP_THRESHOLD, overfit_patience=OVERFIT_PATIENCE,
    min_epochs=MIN_EPOCHS, class_weights=class_weights,
)


## 5. Phase 2 — Ablation (EXP002: +MSF, EXP003: +MSF+CBAM)
Same training recipe as EXP001 (same seed, optimizer, schedule, augmentation) —
the *only* thing that changes is the architecture, so the comparison is fair.


In [ ]:
model_exp002 = build_model("EXP002_resnet50_msf", num_classes=len(CLASSES), pretrained=True)
model_exp002, history_exp002 = fit(
    model_exp002, train_ds, val_ds, exp_name="EXP002_resnet50_msf",
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    overfit_gap_threshold=OVERFIT_GAP_THRESHOLD, overfit_patience=OVERFIT_PATIENCE,
    min_epochs=MIN_EPOCHS, class_weights=class_weights,
)


In [ ]:
model_exp003 = build_model("EXP003_resnet50_msf_cbam", num_classes=len(CLASSES), pretrained=True)
model_exp003, history_exp003 = fit(
    model_exp003, train_ds, val_ds, exp_name="EXP003_resnet50_msf_cbam",
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
    overfit_gap_threshold=OVERFIT_GAP_THRESHOLD, overfit_patience=OVERFIT_PATIENCE,
    min_epochs=MIN_EPOCHS, class_weights=class_weights,
)


In [ ]:
# Training curves — put this figure in your results section.
# Third panel (overfit gap = train_acc - val_acc) is the evidence early stopping
# was actually monitoring, not just the loss curve looking OK.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, h in [("EXP001", history_exp001), ("EXP002", history_exp002), ("EXP003", history_exp003)]:
    axes[0].plot(h["val_loss"], label=name)
    axes[1].plot(h["val_acc"], label=name)
    axes[2].plot(h["overfit_gap"], label=name)
axes[0].set_title("Validation loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
axes[2].axhline(0.15, color="red", linestyle="--", linewidth=1, label="overfit threshold")
axes[2].set_title("Overfit gap (train_acc - val_acc)"); axes[2].set_xlabel("epoch"); axes[2].legend()
plt.tight_layout(); plt.savefig("training_curves.png", dpi=200); plt.show()

for name, h in [("EXP001", history_exp001), ("EXP002", history_exp002), ("EXP003", history_exp003)]:
    print(f"{name}: stopped because '{h['stop_reason']}', best epoch = {h['best_epoch']}")


## 6. Ablation metrics table (Table 2 of the paper)
Evaluated once on the held-out **test** set (never touched during training/model selection).


In [ ]:
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

models = {
    "EXP001_baseline_resnet50": model_exp001,
    "EXP002_resnet50_msf": model_exp002,
    "EXP003_resnet50_msf_cbam": model_exp003,
}

test_predictions = {}
for name, m in models.items():
    m = m.to(device)
    y_true, y_pred, y_prob, paths = get_predictions(m, test_loader, device)
    test_predictions[name] = (y_true, y_pred, y_prob)
    print(f"--- {name} ---")
    print_classwise_report(y_true, y_pred, CLASSES)

ablation_df = build_ablation_table(test_predictions, num_classes=len(CLASSES))
ablation_df


In [ ]:
ablation_df.to_csv("ablation_table.csv")
print(ablation_df.to_markdown())  # paste straight into your paper/thesis draft


## 7. Phase 3a — Calibration (is confidence trustworthy?)
Compares EXP001 (baseline) vs EXP003 (your reference model). If attention improves
calibration even where the accuracy gain over EXP001 is modest, **lead your abstract
with that finding** — it's the more novel, more clinically relevant result.


In [ ]:
calib_reports = {}
bin_stats_dict = {}
for name in ["EXP001_baseline_resnet50", "EXP003_resnet50_msf_cbam"]:
    y_true, y_pred, y_prob = test_predictions[name]
    report, bin_stats = calibration_report(y_true, y_prob, num_classes=len(CLASSES))
    calib_reports[name] = report
    bin_stats_dict[name] = bin_stats
    print(name, report)

plot_reliability_diagram(bin_stats_dict, title="Reliability Diagram: EXP001 vs EXP003",
                          save_path="reliability_diagram.png")
plt.show()


## 8. Phase 3b — Explainability faithfulness (LayerCAM + Deletion/Insertion AOPC)
This is the section that turns "here's a heatmap" into a scientific claim: does
removing the pixels LayerCAM marks important actually hurt the prediction
(deletion, want **low** AOPC), and does revealing only those pixels recover the
prediction (insertion, want **high** AOPC)?

`NUM_FAITHFULNESS_SAMPLES` controls runtime — each sample costs
`num_steps x 2` forward passes, so keep this modest (30–100) unless you have a lot
of GPU time budgeted.


In [ ]:
NUM_FAITHFULNESS_SAMPLES = 50
NUM_STEPS = 20

reference_model = model_exp003.to(device)
target_layer = reference_model.get_target_layer()
cam_engine = LayerCAM(reference_model, target_layer)

faith_report = faithfulness_report(
    reference_model, cam_engine, test_loader, device,
    num_samples=NUM_FAITHFULNESS_SAMPLES, num_steps=NUM_STEPS,
)
print("EXP003 faithfulness:", faith_report)


In [ ]:
# Qualitative LayerCAM gallery — 6 example test images (grab a few per class)
import numpy as np
sample_indices = [0, 400, 800, 1200, 1600, 2000]
sample_indices = [i for i in sample_indices if i < len(test_ds)]

images_np, cams, titles = [], [], []
for idx in sample_indices:
    img_t, label, path = test_ds[idx]
    img_batch = img_t.unsqueeze(0).to(device)
    cam, pred_idx, prob = cam_engine.generate(img_batch)
    denorm = img_t.permute(1, 2, 0).numpy() * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    images_np.append(denorm.clip(0, 1))
    cams.append(cam)
    correct = "correct" if pred_idx == label else "WRONG"
    titles.append(f"true={CLASSES[label]} pred={CLASSES[pred_idx]} ({correct})")

plot_cam_grid(images_np, cams, titles, save_path="layercam_gallery.png")
plt.show()


## 9. Phase 4 — Robustness to acquisition noise (keep lean: one table)


In [ ]:
severities = (1, 2, 3)
robustness_results = evaluate_under_perturbation(
    reference_model, test_loader, device,
    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],
    severities=severities,
)
robustness_df = pd.DataFrame(robustness_results)
robustness_df.to_csv("robustness_table.csv", index=False)
robustness_df


## 10. Phase 4 — Compute cost analysis (a few lines, not a section)

In [ ]:
import time

compute_rows = []
for name, m in models.items():
    m = m.to(device)
    params = count_parameters(m)
    dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
    m.eval()
    with torch.no_grad():
        for _ in range(5):  # warmup
            _ = m(dummy)
        torch.cuda.synchronize() if device.type == "cuda" else None
        t0 = time.time()
        for _ in range(50):
            _ = m(dummy)
        torch.cuda.synchronize() if device.type == "cuda" else None
        elapsed = (time.time() - t0) / 50 * 1000  # ms/image
    compute_rows.append({
        "experiment": name,
        "total_params_M": round(params["total_params"] / 1e6, 2),
        "inference_ms_per_image": round(elapsed, 2),
    })
compute_df = pd.DataFrame(compute_rows).set_index("experiment")
compute_df.to_csv("compute_analysis.csv")
compute_df


## 11. Phase 4 — Failure analysis (4–6 misclassified examples, qualitative)

In [ ]:
y_true, y_pred, y_prob = test_predictions["EXP003_resnet50_msf_cbam"]
misclassified_idx = np.where(y_true != y_pred)[0][:6]

fail_images, fail_cams, fail_titles = [], [], []
for idx in misclassified_idx:
    img_t, label, path = test_ds[idx]
    img_batch = img_t.unsqueeze(0).to(device)
    cam, pred_idx, prob = cam_engine.generate(img_batch, class_idx=int(y_pred[idx]))
    denorm = img_t.permute(1, 2, 0).numpy() * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    fail_images.append(denorm.clip(0, 1))
    fail_cams.append(cam)
    fail_titles.append(f"true={CLASSES[label]} pred={CLASSES[pred_idx]} (p={prob:.2f})")

if len(fail_images) > 0:
    plot_cam_grid(fail_images, fail_cams, fail_titles, save_path="failure_analysis.png")
    plt.show()
else:
    print("No misclassifications in this sample of the test set — nice problem to have.")


## 13. Phase 5 — Statistical significance (multi-seed) and external validation

Optional but strongly recommended before writing the results chapter:

- **Multi-seed ablation**: repeats EXP001/002/003 across several random seeds
  and reports mean ± std plus a paired t-test (EXP003 vs EXP001, EXP003 vs
  EXP002) for every metric. This is what lets you claim an accuracy gap is
  meaningful rather than run-to-run noise.
- **External validation**: evaluates the trained EXP003 model on an
  independent OCT dataset (different source from Kermany) using the same
  metrics + calibration report as the in-distribution test set, so the two
  can sit side-by-side in one table as direct evidence of generalization.

Both are compute-/data-hungry — if Colab time or an external dataset isn't
available, it's fine to run only the multi-seed step (cheaper, and the more
commonly requested of the two by reviewers) and note external validation as
future work rather than skipping the honesty of saying so.


In [ ]:
# NOTE: this repeats full training 3x per experiment (9 runs total) — budget
# Colab GPU time accordingly. Reduce `seeds` to a single value for a quick
# smoke test of the pipeline before committing to the full run.
from trustoct.multiseed import run_multiseed_ablation, format_mean_std_table

per_run_df, summary_df, significance = run_multiseed_ablation(
    train_ds, val_ds, test_ds,
    seeds=(42, 123, 2024),
    epochs=EPOCHS, batch_size=32, lr=1e-4,
    ckpt_dir="/content/checkpoints_multiseed", verbose=True,
)

print("\n=== Per-run results (raw) ===")
display(per_run_df)

print("\n=== Mean ± std (paste into paper) ===")
display(format_mean_std_table(summary_df))

print("\n=== Paired significance tests (EXP003 vs baselines) ===")
for comparison, metric_results in significance.items():
    print(f"\n{comparison}:")
    for metric, res in metric_results.items():
        print(f"  {metric}: t={res['t_stat']:.3f}, p={res['p_value']:.4f}")


In [ ]:
# Fill in `EXTERNAL_DATASET_ROOT` and `CLASS_FOLDER_MAP` for whichever
# independent OCT dataset you have access to (e.g. OCTID, a Duke OCT release,
# or clinic data you have clearance to use). class_folder_map translates that
# dataset's own class-folder names into the Kermany schema
# {CNV, DME, DRUSEN, NORMAL} — any folder not listed here is skipped and
# reported, rather than silently mis-assigned.

from trustoct.external_validation import run_external_validation

EXTERNAL_DATASET_ROOT = "/content/external_oct_dataset"  # <-- set this
CLASS_FOLDER_MAP = {
    # "ExternalFolderName": "KermanySchemaClass",
    # "AMD": "DRUSEN",
    # "Normal": "NORMAL",
}

if CLASS_FOLDER_MAP:
    ext_results = run_external_validation(
        model_exp003, EXTERNAL_DATASET_ROOT, CLASS_FOLDER_MAP,
        image_size=IMAGE_SIZE, num_classes=len(CLASSES),
        save_calibration_plot="/content/external_validation_reliability.png",
    )
else:
    print("CLASS_FOLDER_MAP is empty — set it above once you've chosen an "
          "external dataset and inspected its folder names. Skipping for now.")


## 12. Save everything for your thesis / paper appendix
Bundles all tables, figures, and model checkpoints into one folder you can zip and download.


In [ ]:
import shutil
os.makedirs("trustoct_results", exist_ok=True)
for f in ["ablation_table.csv", "robustness_table.csv", "compute_analysis.csv",
          "training_curves.png", "reliability_diagram.png", "layercam_gallery.png",
          "failure_analysis.png"]:
    if os.path.exists(f):
        shutil.copy(f, f"trustoct_results/{f}")

import json
with open("trustoct_results/calibration_report.json", "w") as f:
    json.dump(calib_reports, f, indent=2)
with open("trustoct_results/faithfulness_report.json", "w") as f:
    json.dump(faith_report, f, indent=2)

shutil.copytree("checkpoints", "trustoct_results/checkpoints", dirs_exist_ok=True)

shutil.make_archive("trustoct_results", "zip", "trustoct_results")
print("Saved trustoct_results.zip — download it from the Colab file browser (left sidebar).")

from google.colab import files
files.download("trustoct_results.zip")


---
## Notes for your thesis write-up
- **Reproducibility**: `set_seed(42)` is called once at the top; the same seed, optimizer (AdamW), schedule (ReduceLROnPlateau), and augmentation pipeline are used for all three experiments — the ablation table isolates the architectural change.
- **No patient leakage**: the split is grouped by patient ID (parsed from the Kermany filename convention), not by image, and `assert_no_patient_leakage` verifies no patient's B-scans appear in more than one of train/val/test. State this explicitly in your methodology section — it's a real, checkable claim, not a formality.
- **Early stopping has two independent triggers**, either one can end training: (1) val-loss plateau (`patience` epochs with no improvement), and (2) an overfitting-gap monitor — `train_acc - val_acc > overfit_gap_threshold` for `overfit_patience` consecutive epochs. The second one matters because val_loss can keep inching down even while the train/val gap quietly widens (common with a pretrained backbone that memorizes quickly), which plateau-only early stopping would miss. Both triggers restore the **best-val-loss checkpoint**, not the epoch training happened to stop at — check `history["stop_reason"]` and `history["best_epoch"]` after each `fit()` call, and report the overfit-gap curve (third panel in the training-curves figure) in your methodology section as evidence you checked for overfitting rather than just eyeballing the loss curve.
- **Why re-split instead of using Kermany's provided test set**: their `test/` folder has only 8 images/class (32 total) — not enough for a stable ROC-AUC or reliability diagram. We stratified-split the ~84k `train/` folder 80/10/10 instead and never touch the test split for model selection.
- **If Colab disconnects mid-training**: `fit()` checkpoints the best model to `checkpoints/{exp_name}_best.pt` after every improving epoch, so you can reload with `trustoct.utils.load_checkpoint` and resume evaluation without retraining from scratch.
- **Full run vs smoke test**: set `MAX_PER_CLASS = None` and `EPOCHS = 25` for the numbers you'll publish; use a small `MAX_PER_CLASS` (e.g. 500) and `EPOCHS = 3` first to confirm the whole notebook runs end-to-end before committing a multi-hour GPU session.
